# Extract the 10-K Items

| Item | Description |
|---|---|
| Goal | Walk the filing index and pull named items out of each annual report through the sec-api.io Extractor: one text file per filing and item, and one status line per attempt. Two runs exist. `sp500_item1` extracted Item 1 for the S&P 500 filings and feeds D1.3.4. `items` extracted Items 1, 1A, 1B, 1C, 7, and 7A for 28,576 filings of the full sample before that run was stopped. |
| Input | `Processed/filings.parquet` from D1.3.2. The Extractor endpoint, `/extractor`, which fetches the filing from SEC.gov by URL. The key `SEC_API_KEY` in `.env` in the D1 folder. |
| Output | In the D1.3 Interim folder, one tree per run: `<run>/item<X>/<cik>/<accessionNo>.txt`, the item text; `<run>/status/extract_<shard>.jsonl`, one line per attempt; `<run>/status/extract_status.parquet`, all status lines of the run in one table. |
| Run | Both runs ran in August 2026 as SLURM array jobs, one shard per task; the notebook kept no cell outputs. Not rerun: the calls go to a paid API, and the text is on disk. Step 9 reads the status logs of the two runs and was executed on 2026-10-31. |

## Step 1. Import packages

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D1 - Data Collection and Processing/D1.3 - SEC Form 10-K Corporate Filings")
NOTEBOOK_NAME = "D1.3.3 - SEC 10-K - Extract 10-K Items"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"  # filings.parquet, written by D1.3.2
ENV_FILE = PROJECT_FOLDER / "D1 - Data Collection and Processing" / ".env"  # holds SEC_API_KEY

## Step 3. Read the run settings and the filing index

Every setting comes from the environment, so the same notebook serves every task of a SLURM array. The run name decides where everything lands, so two runs never share a folder or a log. The rate limit of sec-api is account-wide, about nine requests per second. `TARGET_RATE` is therefore the total across the tasks running at once, and more tasks buy resilience, not speed.

In [ ]:
RUN_NAME = os.environ.get("RUN_NAME", "items")
ITEMS = tuple(i.strip() for i in os.environ.get("ITEMS", "1,1A,1B,1C,7,7A").split(",") if i.strip())
SP500_ONLY = os.environ.get("SP500_ONLY", "0") == "1"
SHARD = int(os.environ.get("SHARD", 0))
N_SHARDS = int(os.environ.get("N_SHARDS", 1))
N_CONCURRENT = int(os.environ.get("N_CONCURRENT", N_SHARDS))  # with --array=0-63%12 these are 64 and 12
TARGET_RATE = float(os.environ.get("TARGET_RATE", 7.0))  # requests per second across all running tasks

RUN_FOLDER = INTERIM_FOLDER / RUN_NAME
STATUS_FOLDER = RUN_FOLDER / "status"
STATUS_FOLDER.mkdir(parents=True, exist_ok=True)

filings = pd.read_parquet(PROCESSED_FOLDER / "filings.parquet")
if SP500_ONLY:
    filings = filings.loc[filings["in_sp500"]]
work = filings.iloc[SHARD::N_SHARDS].reset_index(drop=True)  # shards partition the index and never overlap

SECONDS_BETWEEN_REQUESTS = N_CONCURRENT / TARGET_RATE  # this task's share of the aggregate rate

API_KEY = os.environ.get("SEC_API_KEY")
if not API_KEY and ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():  # one KEY=value per line
        name, _, value = line.strip().partition("=")
        if name == "SEC_API_KEY" and value:
            API_KEY = value.strip().strip('"').strip("'")
assert API_KEY, f"SEC_API_KEY is not set and {ENV_FILE} does not supply it"

print(f"run        : {RUN_NAME}  ->  {RUN_FOLDER}")
print(f"items      : {', '.join(ITEMS)}")
print(f"scope      : {'S&P 500 only' if SP500_ONLY else 'full sample'}")
print(f"shard      : {SHARD} of {N_SHARDS}  ({N_CONCURRENT} running at once)")
print(f"rate       : {TARGET_RATE} req/s aggregate -> {1 / SECONDS_BETWEEN_REQUESTS:.3f} req/s this task")
print(f"filings    : {len(filings):,} in scope, {len(work):,} in this shard")
print(f"calls      : {len(work) * len(ITEMS):,} in this shard, {len(filings) * len(ITEMS):,} in the run")
print(f"projected  : {len(filings) * len(ITEMS) / TARGET_RATE / 3600:.1f} h at {TARGET_RATE} req/s")

## Step 4. Define one API request and one item call

A 429 is flow control, not a failure: the key is blocked only for the rest of the second. The request therefore pauses a quarter second and tries again without spending a retry. Backing off exponentially instead is what collapsed the first run to an eighth of one thread's speed. A 404 saying "filing type not supported" is a permanent property of the filing, so it is raised as its own error and never retried.

In [ ]:
BASE_URL = "https://api.sec-api.io"
MAX_ATTEMPTS = 5
TRANSIENT_STATUS = {408, 425, 500, 502, 503, 504}
RATE_LIMIT_PAUSE = 0.25
MAX_RATE_LIMIT_WAITS = 60

session = requests.Session()
request_log = {"requests": 0, "retries": 0, "rate_limited": 0, "last_request": 0.0}


def secapi_get(url, params):
    """One GET. Pauses on a rate limit, retries a transient failure, raises ValueError for an unsupported filing."""
    last_error = "no attempt made"
    attempt = 0
    waits = 0
    while attempt < MAX_ATTEMPTS:
        wait = SECONDS_BETWEEN_REQUESTS - (time.monotonic() - request_log["last_request"])
        if wait > 0:
            time.sleep(wait)
        request_log["last_request"] = time.monotonic()
        request_log["requests"] += 1
        try:
            response = session.get(url, params=params, headers={"Authorization": API_KEY}, timeout=180)
        except requests.RequestException as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code == 200:
                return response
            if response.status_code == 429:
                waits += 1
                request_log["rate_limited"] += 1
                if waits > MAX_RATE_LIMIT_WAITS:
                    raise RuntimeError(f"still rate limited after {waits} pauses for {url}; the aggregate rate is too high")
                time.sleep(RATE_LIMIT_PAUSE)
                continue
            if response.status_code == 404 and "not supported" in response.text:
                raise ValueError(f"HTTP 404 for {url}: {response.text[:200]}")  # permanent: the Extractor cannot split this filing
            if response.status_code not in TRANSIENT_STATUS:
                raise RuntimeError(f"HTTP {response.status_code} for {url}: {response.text[:200]}")
            last_error = f"HTTP {response.status_code}"
        request_log["retries"] += 1
        attempt += 1
        time.sleep(2.0 ** attempt)
    raise RuntimeError(f"{MAX_ATTEMPTS} attempts failed for {url}: {last_error}")


def get_section(filing_url, item):
    """One 10-K item as text. Item ids are the Extractor's: "1" is Business, "7" is Management's Discussion and Analysis."""
    return secapi_get(f"{BASE_URL}/extractor", {"url": filing_url, "item": item, "type": "text", "token": API_KEY}).text

## Step 5. Define one filing-item

Every attempt returns a status record. `ok` means the section was written, `skipped` that it was already on disk, and `empty` that the item is not in this filing. `unsupported` means the Extractor cannot split the filing, and `error` that a transient failure outlived the retries. Only `error` is retried on a later run, and the write goes to a temporary name first.

In [ ]:
def extract_one(row, item):
    """Extract one item from one filing and return what happened."""
    target = RUN_FOLDER / f"item{item}" / row.cik_unpadded / f"{row.accessionNo}.txt"
    record = {
        "accessionNo": row.accessionNo,
        "cik": row.cik_unpadded,
        "gvkey": row.gvkey,
        "fyear": int(row.fyear),
        "formType": row.formType,
        "item": item,
        "path": str(target),
        "at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if target.exists():
        return {**record, "status": "skipped", "chars": target.stat().st_size}

    try:
        text = get_section(row.filingUrl, item)
    except ValueError as exc:
        return {**record, "status": "unsupported", "chars": 0, "message": str(exc)[:300]}
    except RuntimeError as exc:
        return {**record, "status": "error", "chars": 0, "message": str(exc)[:300]}

    if not text or not text.strip():
        return {**record, "status": "empty", "chars": 0}

    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(".txt.partial")
    partial.write_text(text)
    partial.replace(target)
    return {**record, "status": "ok", "chars": len(text)}

## Step 6. Walk the shard

Status lines are appended and flushed as they happen, so a task killed at the wall clock loses only the call in flight. The set of filing-items already done is read from every shard's log in the run. A run sharded differently later therefore skips the work already done.

In [ ]:
TERMINAL = {"ok", "skipped", "empty", "unsupported"}  # every status except error is final

status_path = STATUS_FOLDER / f"extract_{SHARD:03d}.jsonl"

done = set()
for log in sorted(STATUS_FOLDER.glob("extract_*.jsonl")):
    for line in log.read_text().splitlines():
        if line.strip():
            row = json.loads(line)
            if row["status"] in TERMINAL:
                done.add((row["accessionNo"], row["item"]))

counts = {"ok": 0, "skipped": 0, "empty": 0, "unsupported": 0, "error": 0}
todo = [(r, item) for r in work.itertuples() for item in ITEMS if (r.accessionNo, item) not in done]
print(f"{len(done):,} filing-items already recorded, {len(todo):,} still to fetch")

started = datetime.now(timezone.utc)
with status_path.open("a") as log:
    for n, (row, item) in enumerate(todo, 1):
        record = extract_one(row, item)
        log.write(json.dumps(record) + "\n")
        log.flush()
        counts[record["status"]] += 1
        if n % 1000 == 0:
            secs = (datetime.now(timezone.utc) - started).total_seconds()
            rate = n / secs if secs else 0
            eta = (len(todo) - n) / rate / 3600 if rate else 0
            print(f"  {n:,}/{len(todo):,}  {rate:.2f} req/s  eta {eta:.1f} h  " + " ".join(f"{k}={v:,}" for k, v in counts.items() if v))

secs = (datetime.now(timezone.utc) - started).total_seconds()
print()
print("totals:", " ".join(f"{k}={v:,}" for k, v in counts.items()))
print(f"{len(todo):,} attempts in {secs / 60:.1f} min = {len(todo) / secs if secs else 0:.2f} req/s for this task")
print(f"{request_log['requests']:,} requests, {request_log['retries']:,} retries, {request_log['rate_limited']:,} rate-limit pauses")

## Step 7. Consolidate the status log

Only an unsharded pass consolidates. Under a SLURM array every task runs this notebook and finishes at a different time, so each would overwrite the shared file. Running the notebook once more with `N_SHARDS=1` after the array finishes writes the consolidated table. That costs nothing, because Step 6 then finds nothing to do.

In [ ]:
IS_AGGREGATE = N_SHARDS == 1

if IS_AGGREGATE:
    frames = [pd.read_json(p, lines=True) for p in sorted(STATUS_FOLDER.glob("extract_*.jsonl"))]
    status = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    status.to_parquet(STATUS_FOLDER / "extract_status.parquet", index=False)
    print(f"consolidated {len(frames)} shard logs -> {STATUS_FOLDER / 'extract_status.parquet'}")
else:
    status = pd.read_json(status_path, lines=True) if status_path.exists() else pd.DataFrame()
    print(f"shard {SHARD}: reporting on this shard's own log only; run again with N_SHARDS=1 to consolidate")
print()

assert len(status), "no status rows at all: Step 6 wrote nothing"
status["item"] = status["item"].astype(str)  # read_json turns "1" and "7" into numbers
status = status.drop_duplicates(["accessionNo", "item"], keep="last")  # the final state of a filing-item is what counts

pivot = status.pivot_table(index="item", columns="status", values="accessionNo", aggfunc="count", fill_value=0)
print(pivot.to_string())
print()
got = status.loc[status["status"].isin(["ok", "skipped"])]
if len(got):
    print(f"characters extracted: {got['chars'].sum() / 1e9:.3f} billion")
    print(got.groupby("item")["chars"].describe()[["count", "mean", "50%", "max"]].to_string())

## Step 8. Check the run

Two stop conditions and three descriptions. Under an array each task checks its own shard; the consolidation pass checks the whole run.

- No `error` rows remain.
- Every file recorded as written is on disk.
- How many filing-items were attempted, against how many the run covers.
- The share of empty answers by disclosure era. Item 7A exists from 1997, Items 1A and 1B from December 2005, Item 1C from December 2023, so a high empty rate before an item's own cut is the rule, not a fault.
- Firm-year coverage, and any filing the Extractor could not split at all.

In [ ]:
errors = status.loc[status["status"] == "error"]
assert len(errors) == 0, f"{len(errors):,} extractions failed after retries; run again to resume. First: {errors.iloc[0]['accessionNo']} / item {errors.iloc[0]['item']}"
print("1) no unresolved transient errors")

claimed = status.loc[status["status"] == "ok", "path"]
missing = [p for p in claimed if not Path(p).exists()]
assert not missing, f"{len(missing):,} files recorded as written are not on disk"
print(f"2) all {len(claimed):,} written files are on disk")

covered = status.drop_duplicates(["accessionNo", "item"])
expected = len(work) * len(ITEMS) if not IS_AGGREGATE else len(filings) * len(ITEMS)
print(f"3) {len(covered):,} of {expected:,} filing-items attempted ({len(covered) / expected:.1%})")

era = pd.cut(status["fyear"], [0, 1996, 2004, 2022, 2100], labels=["<1997", "1997-2004", "2005-2022", "2023+"])
empty_rate = pd.crosstab(era, status["item"], values=status["status"].eq("empty"), aggfunc="mean")
print("4) share of extractions returning nothing, by disclosure era:")
print(empty_rate.map("{:.1%}".format).to_string())

panel = got.drop_duplicates(["gvkey", "fyear", "item"])
print(f"\n5) firm-year coverage: {panel['gvkey'].nunique():,} firms, {len(panel.drop_duplicates(['gvkey', 'fyear'])):,} firm-years")

unsupported = status.loc[status["status"] == "unsupported"].drop_duplicates("accessionNo")
if len(unsupported):
    print(f"\n   {len(unsupported):,} filings the Extractor could not split at all; every bare-CIK URL was repaired in D1.3.2, so these are new cases")
    print(unsupported.groupby("fyear").size().to_string())
else:
    print("\n   no unsupported filings: the URL repair in D1.3.2 held")

## Step 9. Read the status logs of the completed runs

Evidence of the two runs as they happened. This cell reads only the saved logs under the Interim folder, so it can run at any time without an API call. The paths in the logs are those of the machine that ran them. The files on disk are therefore matched by their path below the run folder.

In [1]:
filings = pd.read_parquet(PROCESSED_FOLDER / "filings.parquet")

for run_folder in sorted(p for p in INTERIM_FOLDER.iterdir() if (p / "status").is_dir()):
    logs = sorted((run_folder / "status").glob("extract_*.jsonl"))
    if not logs:
        continue
    status = pd.concat([pd.read_json(p, lines=True) for p in logs], ignore_index=True)
    status = status.drop_duplicates(["accessionNo", "item"], keep="last")
    status["item"] = status["item"].astype(str)  # read_json turns "1" and "7" into numbers
    items = sorted(status["item"].unique(), key=lambda s: (len(s), s))
    in_scope = filings.loc[filings["accessionNo"].isin(status["accessionNo"])]
    print("=" * 100)
    print(f"run {run_folder.name}: {len(logs)} shard logs, {len(status):,} filing-items, items {', '.join(items)}")
    print(f"  filings covered: {status['accessionNo'].nunique():,} of {len(filings):,} in the index "
          f"({int(in_scope['in_sp500'].sum()):,} of them S&P 500)")
    print(f"  errors left: {int(status['status'].eq('error').sum()):,}")
    print()
    print(status.pivot_table(index="item", columns="status", values="accessionNo", aggfunc="count", fill_value=0).to_string())
    got = status.loc[status["status"].isin(["ok", "skipped"])]
    print(f"\n  characters extracted: {got['chars'].sum() / 1e9:.3f} billion")
    # files on disk, matched on item<X>/<cik>/<accessionNo>.txt below the run folder
    present = set()
    for item_folder in run_folder.glob("item*"):
        present.update(p.relative_to(run_folder).as_posix() for p in item_folder.rglob("*.txt"))
    claimed = set("/".join(Path(p).parts[-3:]) for p in status.loc[status["status"] == "ok", "path"])
    print(f"  files on disk: {len(present):,}; recorded as written: {len(claimed):,}; recorded but absent: {len(claimed - present):,}")
    era = pd.cut(status["fyear"], [0, 1996, 2004, 2022, 2100], labels=["<1997", "1997-2004", "2005-2022", "2023+"])
    print("\n  share of empty answers by disclosure era:")
    print(pd.crosstab(era, status["item"], values=status["status"].eq("empty"), aggfunc="mean").map("{:.1%}".format).to_string())
    print()

run items: 8 shard logs, 171,433 filing-items, items 1, 7, 1A, 1B, 1C, 7A
  filings covered: 28,576 of 158,396 in the index (3,600 of them S&P 500)
  errors left: 0

status  empty     ok  skipped
item                         
1         501  27870      205
1A      13677  14755      143
1B      14311  14119      143
1C      27356   1199       16
7         411  27959      200
7A       3458  24910      200

  characters extracted: 4.159 billion
  files on disk: 111,719; recorded as written: 110,812; recorded but absent: 0

  share of empty answers by disclosure era:
item          1     1A      1B      1C     7     7A
fyear                                              
<1997      3.9%  99.8%  100.0%  100.0%  3.3%  99.1%
1997-2004  2.8%  99.3%  100.0%  100.0%  1.9%  14.0%
2005-2022  0.8%   3.2%    7.1%  100.0%  1.0%   4.2%
2023+      0.0%   0.5%    0.5%    4.6%  0.0%   1.3%

run sp500_item1: 4 shard logs, 22,299 filing-items, items 1
  filings covered: 22,299 of 158,396 in the index (22,299 